In [14]:
import pandas as pd
import numpy as np
import random 



In [15]:
class MySVM:
    def __init__(self, n_iter = 10, learning_rate = 0.001, C = 1, 
                 sgd_sample = None, random_state = 42):
        self.n_iter = n_iter
        self.learning_rate = learning_rate 
        self.C = C

        self.weights = None
        self.b = None
        
        self.sgd_sample = sgd_sample
        self._seed(random_state)

    def _seed(self, random_state):
        self.random_state = random_state 
        random.seed(self.random_state)
        
    def __str__(self):
        return f"MySVM class: n_iter={self.n_iter}, learning_rate={self.learning_rate}"

    def fit(self, X: pd.DataFrame, y: pd.Series, verbose=False):
        y = y.copy()
        y.loc[y == 0] = -1

        self.weights = pd.Series(1, index=X.columns)
        self.b = 1

        for i in range(1, self.n_iter + 1):

            if self.sgd_sample is not None:
                if isinstance(self.sgd_sample, int):
                    sgd_sample = self.sgd_sample
                elif isinstance(self.sgd_sample, float):
                    sgd_sample = round(self.sgd_sample * len(X))
                    
                sample_rows_idx = random.sample(range(X.shape[0]), sgd_sample)
            
                X_hat = []
                y_hat = []
                indexes_hat = []
                for j in sample_rows_idx:
                    X_hat.append(X.iloc[j])
                    y_hat.append(y.iloc[j])
                    indexes_hat.append(X.index[j])
                X_hat = pd.concat(X_hat, axis=1).T
                y_hat = pd.Series(y_hat, index= indexes_hat)
            else:
                X_hat = X
                y_hat = y
                
            for index, row in X_hat.iterrows():
                y_index = y_hat.loc[index]
                if y_index * (row.dot(self.weights) + self.b) >= 1:
                    gradient_weights = 2 * self.weights
                    gradient_bias = 0
                else:
                    gradient_weights = 2 * self.weights - self.C * y_index * row
                    gradient_bias = - self.C * y_index
                
                self.weights -= self.learning_rate * gradient_weights
                self.b -= self.learning_rate * gradient_bias
    
                loss = np.sum(self.weights ** 2) + self.C / len(X) * np.sum( (1 - y * (X.dot(self.weights) + self.b)).map(lambda x: max(0, x)))
            
    def get_coef(self):
        return (self.weights.values, self.b)

    def predict(self, X: pd.DataFrame):
        y = np.sign(X.dot(self.weights) + self.b)
        y.loc[y == -1] = 0
        y = y.map(int)
        return y
        

